# CDG folder - code walkthrough

Objectif: expliquer le raisonnement, le fonctionnement du code, et le calcul derriere les courbes generees.
Ce notebook decrit le role de chaque module et montre comment lancer les principaux pipelines.

## 1) Vue d ensemble du projet

Modules principaux:
- models/altman_scoring.py: Z''-score Altman pour scoring PE
- pipelines/scoring_pipeline.py: orchestration scoring + courbes
- pipelines/monte_carlo.py: stress testing Monte Carlo + courbes
- models/binomial_option.py: arbre binomial CRR (options reelles)
- market_watch/veille_sectorielle.py: veille de marche + indicateurs techniques
- data/: donnees brutes, synthetiques, et traitees
- reports/outputs/: exports PNG des courbes

Flux logique:
donnees -> calculs (ratios, simulations) -> rapports -> courbes.

## 2) Setup rapide

```bash
python -m venv .venv
.venv\Scripts\activate
pip install -r requirements.txt
```

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "models").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from models.altman_scoring import (
    generate_synthetic_data,
    compute_altman_ratios,
    compute_z_score,
    build_scoring_report,
    summarize_risk_distribution,
)
from pipelines.scoring_pipeline import (
    run_scoring_pipeline,
    plot_risk_distribution,
    plot_ranking_chart,
    plot_ratios_heatmap,
)
from pipelines.monte_carlo import (
    run_monte_carlo,
    compute_risk_metrics,
    plot_fan_chart,
    plot_return_distribution,
)
from models.binomial_option import (
    build_binomial_tree,
    plot_binomial_tree,
    sensitivity_analysis,
)

## 3) Altman Z'' scoring (models/altman_scoring.py)

Ce module implemente le Z''-score pour entreprises non cotees.
Formule:
$$Z^{\prime\prime} = 6.56 X_1 + 3.26 X_2 + 6.72 X_3 + 1.05 X_4$$

Ratios:
- $X_1 = \frac{workingcapital}{totalassets}$ (liquidite)
- $X_2 = \frac{retainedearnings}{totalassets}$ (auto financement)
- $X_3 = \frac{ebit}{totalassets}$ (rentabilite)
- $X_4 = \frac{bookequity}{totalliabilities}$ (solvabilite)

Seuils: Safe > 2.60, Zone grise 1.10-2.60, Distress <= 1.10.

Fonctions et logique:
- `generate_synthetic_data(n, seed)`: cree un jeu synthetique avec ratios financiers plausibles.
- `compute_altman_ratios(df)`: calcule X1 a X4 avec un epsilon pour eviter la division par zero.
- `compute_z_score(df)`: applique la formule et ajoute `z_score` + `risk_zone`.
- `build_scoring_report(df)`: trie par `z_score` croissant et arrondit les colonnes clefs.
- `summarize_risk_distribution(report)`: compte les zones de risque et calcule les pourcentages.

In [ ]:
df_raw = generate_synthetic_data(n=20, seed=42)
df_ratios = compute_altman_ratios(df_raw)
df_scored = compute_z_score(df_ratios)
report = build_scoring_report(df_scored)

report.head(5)

summary = summarize_risk_distribution(report)
summary

## 4) Scoring pipeline (pipelines/scoring_pipeline.py)

Fonctions:
- `ingest_data()`: choisit donnees synthetiques ou CSV, verifie les colonnes.
- `run_scoring_pipeline(df)`: enchaine ratios -> Z'' -> rapport.
- `plot_risk_distribution(report)`: histogramme des Z'' + camembert par zone.
- `plot_ranking_chart(report, n)`: barplot horizontal des n entreprises les plus a risque.
- `plot_ratios_heatmap(report, n)`: heatmap des ratios X1-X4 pour les n premieres.
- `export_results(report)`: export CSV dans data/processed/scoring_report.csv.

Calcul des courbes:
- Histogramme: distribution de `z_score` avec zones colorees par seuils.
- Camembert: comptage des `risk_zone`.
- Ranking: barres colorees selon les seuils; lignes verticales aux seuils.
- Heatmap: valeurs X1-X4 en couleurs (palette divergente).

Note: `risk_zone` inclut des icones dans le module Altman, alors que les couleurs du camembert utilisent des labels simples. Si les couleurs paraissent incorrectes, alignez les labels ou le mapping.

In [ ]:
df = generate_synthetic_data(n=100, seed=42)
report = run_scoring_pipeline(df)

plot_risk_distribution(report)
plot_ranking_chart(report, n=20)
plot_ratios_heatmap(report, n=30)

report.head(5)

## 5) Monte Carlo stress testing (pipelines/monte_carlo.py)

Modele GBM utilise pour les trajectoires:
$$S_{t+dt} = S_t \cdot \exp((\mu - 0.5\sigma^2)dt + \sigma \sqrt{dt} Z)$$
avec $Z \sim \mathcal{N}(0,1)$.

Fonctions:
- `run_monte_carlo(...)`: genere une matrice de trajectoires (n_steps+1, n_simulations).
- `compute_risk_metrics(paths, S0)`: calcule stats, VaR et CVaR sur les rendements terminaux.
- `plot_fan_chart(...)`: trace un nuage de trajectoires + percentiles (P5, P25, P50, P75, P95).
- `plot_return_distribution(...)`: histogramme des rendements + KDE + lignes VaR/CVaR.

Def de VaR/CVaR sur les rendements finaux $R$:
- $VaR_c = percentile_{1-c}(R)$
- $CVaR_c = E[R | R \le VaR_c]$

In [ ]:
params = {
    "S0": 100_000_000,
    "mu": 0.10,
    "sigma": 0.22,
    "T": 3.0,
    "dt": 1 / 12,
    "n_simulations": 2000,
    "seed": 42,
}

paths = run_monte_carlo(**params)
metrics = compute_risk_metrics(paths, params["S0"], confidence_levels=[0.95, 0.99])

metrics["VaR"], metrics["CVaR"]

plot_fan_chart(paths, params["T"], params["dt"], metrics, n_display=200, save=True)
plot_return_distribution(paths, params["S0"], metrics, save=True)

## 6) Binomial option valuation (models/binomial_option.py)

Modele CRR (Cox-Ross-Rubinstein):
$$dt = T/N$$
$$u = e^{\sigma \sqrt{dt}}$$
$$d = 1/u$$
$$p = \frac{e^{r dt} - d}{u - d}$$

Backward induction (option americaine):
$$V_{i,j} = \max(\text{intrinsic}, e^{-r dt}(p V^{up}_{i+1,j} + (1-p) V^{down}_{i+1,j+1}))$$

Fonctions:
- `build_binomial_tree(...)`: construit l arbre de l actif et l arbre de l option.
- `plot_binomial_tree(result)`: dessine les noeuds (actif et option) pour N<=6.
- `sensitivity_analysis(...)`: calcule le prix sur une plage de parametres.

In [ ]:
params_opt = {
    "S0": 100_000_000,
    "K": 80_000_000,
    "T": 3,
    "r": 0.035,
    "sigma": 0.25,
    "N": 5,
    "option_type": "call",
}

result = build_binomial_tree(**params_opt)
result["price"]

plot_binomial_tree(result, max_steps=5, save_path=None)

sigma_range = np.linspace(0.10, 0.50, 9)
sensitivity = sensitivity_analysis(params_opt, "sigma", sigma_range)
sensitivity

## 7) Market watch and technical indicators (market_watch/veille_sectorielle.py)

Fonctions principales:
- `fetch_market_data(tickers, period, interval)`: telecharge OHLCV via yfinance.
- `fetch_fundamentals(tickers)`: recupere marketCap, PE, dividendYield, beta, etc.
- `compute_technical_indicators(df)`:
  - MA20 et MA50: moyennes mobiles
  - Daily_Return: pct_change du close
  - Volatility_20d: std(Daily_Return, 20) * sqrt(252)
  - RSI(14): $RSI = 100 - 100/(1+RS)$, avec $RS=avggain/avgloss$
  - Bollinger: $BB_{upper} = MA_{20} + 2\sigma_{20}$, $BB_{lower} = MA_{20} - 2\sigma_{20}$
  - Trend_Signal: MA20 vs MA50
- `compute_sector_trend_score(all_data)`:
  - ret_1m = pct_change(21), ret_1w = pct_change(5)
  - score_ret = clip(ret_1m * 200, -50, 50)
  - score_rsi = rsi - 50
  - trend_score = score_ret + score_rsi
- `plot_price_dashboard(all_data)`:
  - Courbe du prix + MA + bandes de Bollinger
  - Volume colore selon le signe du rendement
  - RSI avec seuils 30/70
- `plot_sector_heatmap(score_df)`:
  - Heatmap des ret_1w, ret_1m, rsi_14, trend_score
- `export_market_data(...)`: export CSV pour OHLCV et scores

Note: cette partie demande une connexion internet et yfinance.

In [ ]:
RUN_MARKET_WATCH = False

if RUN_MARKET_WATCH:
    try:
        from market_watch.veille_sectorielle import (
            fetch_market_data,
            compute_technical_indicators,
            compute_sector_trend_score,
            plot_price_dashboard,
            plot_sector_heatmap,
        )
    except Exception as exc:
        print(f"Import error: {exc}")
    else:
        tickers = ["ENEL.MI", "VIE.PA"]
        data = fetch_market_data(tickers, period="6mo", interval="1d")
        for t in data:
            data[t] = compute_technical_indicators(data[t])
        score_df = compute_sector_trend_score(data)
        plot_price_dashboard(data, save=True)
        plot_sector_heatmap(score_df, save=True)
        score_df.head()

## 8) Outputs et courbes generees

Principaux fichiers produits:
- reports/outputs/scoring_distribution.png: histogramme + camembert
- reports/outputs/scoring_ranking.png: barplot classement
- reports/outputs/scoring_heatmap.png: heatmap ratios X1-X4
- reports/outputs/mc_fan_chart.png: fan chart Monte Carlo
- reports/outputs/mc_return_distribution.png: distribution des rendements
- reports/outputs/binomial_tree.png (si sauvegarde)
- reports/outputs/veille_*.png et veille_sector_heatmap.png
- data/processed/scoring_report.csv
- data/raw/*_ohlcv.csv et data/raw/sector_trend_scores.csv